# 주식 알리미 봇 - 자격증명 생성 (브라우저에서 1회만)

설치 필요 없습니다. 아래 셀 왼쪽의 **▶ 실행** 버튼을 누르고 안내대로 입력하세요.

1. API ID / API HASH / 전화번호 / 봇 토큰 입력
2. 텔레그램으로 오는 **5자리 코드** 입력
3. 텔레그램에서 봇에게 `안녕` 한 번 보내고 Enter
4. 맨 아래 출력된 값들을 **GitHub Secrets** 에 복사해 넣기

> 입력값은 이 브라우저 세션에서만 쓰이고 저장되지 않습니다. 끝나면 `런타임 > 런타임 초기화` 하세요.

In [ ]:
!pip -q install telethon requests
import asyncio, requests
from telethon import TelegramClient
from telethon.sessions import StringSession

api_id   = int(input('API ID (숫자): ').strip())
api_hash = input('API HASH: ').strip()
phone    = input('전화번호 (+82...): ').strip()
bot_token= input('봇 토큰: ').strip()

async def run():
    print('\n[1/2] 로그인 중... 텔레그램으로 오는 5자리 코드를 입력하세요.')
    client = TelegramClient(StringSession(), api_id, api_hash)
    await client.start(phone=phone)
    s = client.session.save()
    me = await client.get_me()
    await client.disconnect()
    print('  로그인 완료:', me.first_name)
    input('\n[2/2] 텔레그램에서 봇에게 \'안녕\' 을 보낸 뒤 Enter... ')
    r = requests.get(f'https://api.telegram.org/bot{bot_token}/getUpdates', timeout=30).json()
    chat_id = None
    for u in r.get('result', []):
        c = (u.get('message') or u.get('channel_post') or {}).get('chat', {})
        if c.get('id'):
            chat_id = c['id']
            if c['id'] > 0: break
    print('\n' + '='*60)
    print(' GitHub Secrets 에 넣을 값 (이름 = 값)')
    print('='*60)
    print('TG_API_ID         =', api_id)
    print('TG_API_HASH       =', api_hash)
    print('TG_PHONE          =', phone)
    print('BOT_TOKEN         =', bot_token)
    print('CHAT_ID           =', chat_id)
    print('TG_SESSION_STRING =', s)
    print('='*60)
    print('(GEMINI_API_KEY 는 따로 발급받은 값을 넣으세요)')

await run()